# PHASE 3: TỐI ƯU HÓA MÔ HÌNH NETWORK
## CPU Only - Simple & Stable

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import lightgbm as lgb
import xgboost as xgb
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

os.makedirs('Phase3_Network_Models', exist_ok=True)
os.makedirs('Phase3_Network_Data', exist_ok=True)

print("=" * 60)
print("PHASE 3: TỐI ƯU HÓA MÔ HÌNH NETWORK")
print("=" * 60)

PHASE 3: TỐI ƯU HÓA MÔ HÌNH NETWORK


## BƯỚC 1: LOAD DỮ LIỆU VÀ MODEL

In [2]:
print("\nBƯỚC 1: LOAD DỮ LIỆU VÀ MODEL")
print("-" * 40)

# Load processed data from Phase 1
X_train = pd.read_csv('Phase1_Network_Data/X_train_processed.csv')
X_test = pd.read_csv('Phase1_Network_Data/X_test_processed.csv')
y_train = pd.read_csv('Phase1_Network_Data/y_train_processed.csv').iloc[:, 0]
y_test = pd.read_csv('Phase1_Network_Data/y_test_processed.csv').iloc[:, 0]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Load best model info from Phase 2
with open('Phase2_Network_Models/best_model_info.pkl', 'rb') as f:
    best_model_info = pickle.load(f)

best_model_name = best_model_info['best_model_name']
print(f"Best model từ Phase 2: {best_model_name}")
print(f"F1-Score hiện tại: {best_model_info['test_f1_score']:.4f}")


BƯỚC 1: LOAD DỮ LIỆU VÀ MODEL
----------------------------------------
Train: (82332, 42), Test: (175341, 42)
Best model từ Phase 2: XGBoost
F1-Score hiện tại: 0.9028


## BƯỚC 2: THIẾT LẬP OPTIMIZATION

In [3]:
print("\nBƯỚC 2: THIẾT LẬP OPTIMIZATION")
print("-" * 40)

if 'XGBoost' in best_model_name:
    print("Optimizing XGBoost (CPU)...")
    base_model = xgb.XGBClassifier(random_state=42, n_jobs=-1)
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [20],
        'learning_rate': [0.1],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0]
    }
elif 'LightGBM' in best_model_name:
    print("Optimizing LightGBM (CPU)...")
    base_model = lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [20],
        'learning_rate': [0.1],
        'num_leaves': [31, 50],
        'subsample': [0.8, 1.0]
    }
else:
    print(f"Model {best_model_name} không cần optimization")
    model_filename = best_model_info['model_filename']
    with open(f'Phase2_Network_Models/{model_filename}', 'rb') as f:
        optimized_model = pickle.load(f)
    param_grid = None


BƯỚC 2: THIẾT LẬP OPTIMIZATION
----------------------------------------
Optimizing XGBoost (CPU)...


## BƯỚC 3: RANDOMIZED SEARCH

In [4]:
if param_grid is not None:
    print("\nBƯỚC 3: RANDOMIZED SEARCH")
    print("-" * 40)
    
    cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
    
    random_search = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=param_grid,
        n_iter=6,
        cv=cv,
        scoring='f1_weighted',
        n_jobs=-1,
        verbose=1,
        random_state=42
    )
    
    print("Bắt đầu Randomized Search...")
    print("Total fits: 6 × 2 = 12")
    print("Estimated time: 2-3 minutes")
    
    start_time = time.time()
    random_search.fit(X_train, y_train)
    search_time = time.time() - start_time
    
    print(f"\nHoàn thành trong {search_time:.2f}s ({search_time/60:.1f} phút)")
    print(f"Best params: {random_search.best_params_}")
    print(f"Best CV score: {random_search.best_score_:.4f}")
    
    optimized_model = random_search.best_estimator_
else:
    print("\nBƯỚC 3: SKIP OPTIMIZATION")
    print("-" * 40)
    print("Sử dụng model từ Phase 2")


BƯỚC 3: RANDOMIZED SEARCH
----------------------------------------
Bắt đầu Randomized Search...
Total fits: 6 × 2 = 12
Estimated time: 2-3 minutes
Fitting 2 folds for each of 6 candidates, totalling 12 fits

Hoàn thành trong 18.24s (0.3 phút)
Best params: {'subsample': 0.8, 'n_estimators': 100, 'max_depth': 20, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
Best CV score: 0.9786


## BƯỚC 4: ĐÁNH GIÁ

In [5]:
print("\nBƯỚC 4: ĐÁNH GIÁ MODEL")
print("-" * 40)

y_pred = optimized_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

print(f"\nSo sánh:")
print(f"  Phase 2: {best_model_info['test_f1_score']:.4f}")
print(f"  Phase 3: {f1:.4f}")
print(f"  Improvement: {(f1 - best_model_info['test_f1_score']):.4f}")


BƯỚC 4: ĐÁNH GIÁ MODEL
----------------------------------------
Accuracy: 0.8991
Precision: 0.9179
Recall: 0.8991
F1-Score: 0.9017

So sánh:
  Phase 2: 0.9028
  Phase 3: 0.9017
  Improvement: -0.0012


## BƯỚC 5: LƯU MODEL

In [6]:
print("\nBƯỚC 5: LƯU MODEL")
print("-" * 40)

model_filename = 'network_' + best_model_name.lower().replace(" ", "_") + '_optimized.pkl'
with open(f'Phase3_Network_Models/{model_filename}', 'wb') as f:
    pickle.dump(optimized_model, f)

optimized_model_info = {
    'model_name': best_model_name,
    'test_accuracy': accuracy,
    'test_precision': precision,
    'test_recall': recall,
    'test_f1_score': f1,
    'model_filename': model_filename
}

with open('Phase3_Network_Models/optimized_model_info.pkl', 'wb') as f:
    pickle.dump(optimized_model_info, f)

print("✅ Đã lưu optimized model")


BƯỚC 5: LƯU MODEL
----------------------------------------
✅ Đã lưu optimized model


## BƯỚC 5: TEST TRÊN RAW DATASET

In [7]:
print("\nBƯỚC 5: TEST TRÊN RAW DATASET")
print("-" * 40)

# Load raw test data
test_raw = pd.read_csv('Network/UNSW_NB15_testing-set.csv')
print(f"Raw test data: {test_raw.shape}")

# Drop unnecessary columns (same as Phase 1)
cols_to_drop = ['id', 'attack_cat', 'label']
cols_to_drop = [col for col in cols_to_drop if col in test_raw.columns]

# Separate features and target
X_test_raw = test_raw.drop(cols_to_drop, axis=1)
y_test_raw = test_raw['label']

# Load preprocessing artifacts
with open('Phase1_Network_Models/network_feature_encoders.pkl', 'rb') as f:
    feature_encoders = pickle.load(f)
with open('Phase1_Network_Models/network_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

# Apply encoding to categorical columns
X_test_encoded = X_test_raw.copy()
for col, encoder in feature_encoders.items():
    if col in X_test_encoded.columns:
        test_values = X_test_encoded[col].astype(str).values
        class_to_idx = {cls: idx for idx, cls in enumerate(encoder.classes_)}
        X_test_encoded[col] = np.array([class_to_idx.get(val, 0) for val in test_values])

# Apply scaling (CRITICAL FIX)
X_test_scaled = scaler.transform(X_test_encoded)

print(f"Features after encoding and scaling: {X_test_scaled.shape}")

# Predict with optimized model
y_pred_raw = optimized_model.predict(X_test_scaled)

# Evaluate
accuracy_raw = accuracy_score(y_test_raw, y_pred_raw)
precision_raw = precision_score(y_test_raw, y_pred_raw, average='weighted', zero_division=0)
recall_raw = recall_score(y_test_raw, y_pred_raw, average='weighted', zero_division=0)
f1_raw = f1_score(y_test_raw, y_pred_raw, average='weighted', zero_division=0)

print(f"\nKết quả test trên RAW dataset:")
print(f"Accuracy: {accuracy_raw:.4f}")
print(f"Precision: {precision_raw:.4f}")
print(f"Recall: {recall_raw:.4f}")
print(f"F1-Score: {f1_raw:.4f}")

print(f"\nSo sánh Processed vs Raw:")
print(f"  Processed F1: {f1:.4f}")
print(f"  Raw F1: {f1_raw:.4f}")
print(f"  Difference: {abs(f1 - f1_raw):.4f}")


BƯỚC 5: TEST TRÊN RAW DATASET
----------------------------------------
Raw test data: (175341, 45)
Features after encoding and scaling: (175341, 42)

Kết quả test trên RAW dataset:
Accuracy: 0.8991
Precision: 0.9179
Recall: 0.8991
F1-Score: 0.9017

So sánh Processed vs Raw:
  Processed F1: 0.9017
  Raw F1: 0.9017
  Difference: 0.0000


In [8]:
print("\n" + "=" * 60)
print("HOÀN THÀNH PHASE 3!")
print("=" * 60)
print(f"Model: {best_model_name}")
print(f"F1 (Processed): {f1:.4f}")
print(f"F1 (Raw): {f1_raw:.4f}")


HOÀN THÀNH PHASE 3!
Model: XGBoost
F1 (Processed): 0.9017
F1 (Raw): 0.9017
